<img src="https://raw.githubusercontent.com/IDEALLab/EngiOpt/codex/dcc26-workshop-notebooks/workshops/dcc26/assets/engibench_logo.png" width="560"/>

# Notebook 02 — Evaluating your generated designs

*A guided tour, not an exercise sheet. Just run the cells top to bottom and read the prose between them.*

> **Colab users:** click **File ➜ Save a copy in Drive** before editing so your changes persist.

## Where we are in the workshop

By the end of **Notebook 01** you had a stack of generated designs sitting in a folder. They *look* like beams — sort of. The optimiser's designs look sharper. The generator's designs look blurrier. Do those two sentences tell us anything useful?

Not really. A picture tells you a design is *plausible*. A picture does not tell you:

- Whether the design **obeys the physical rules** (the constraints).
- Whether the design is **actually stiff** under the load it was designed for.
- Whether the model is producing **varied** designs or secretly copying one.
- Whether a generated design is **a better starting point** for the classical optimiser than a blank slate.

Those are four engineering questions, not four ML questions, and every one of them has an answer on the same `problem` object we used in Notebook 00. This notebook just asks them, one by one, and reports what the benchmark says.

## What this notebook is — and isn't

This is *not* a survey of every generative-model metric in the literature. There are many (MMD, DPP, FID, coverage, precision/recall, …) and a real benchmark report would use several. We're going to stick with the four questions above because each one maps **directly** onto a single method on `problem` that you already met in Notebook 00:

| Engineering question                        | Who answers it      |
|----------------------------------------------|---------------------|
| Does the design obey the rules?              | `problem.check_constraints(...)` |
| Does the design actually work?               | `problem.simulate(...)` |
| Is the model producing varied designs?       | *(one line of NumPy)* |
| Does it help the classical optimiser?        | `problem.optimize(...)` |

That's the pedagogical point: **the benchmark's evaluation interface is the same interface we've been using the whole time.** Once the problem's methods are nailed down, scoring an ML method against them is mechanical.

## Install dependencies (Colab / fresh env only)

Skip this if your local environment already has `engibench` and `engiopt` installed.

In [ ]:
import subprocess, sys

IN_COLAB = "google.colab" in sys.modules
FORCE_INSTALL = False  # flip to True to force install locally

if IN_COLAB or FORCE_INSTALL:
    def _pip(pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])
    _pip(["engibench[all]", "matplotlib", "tqdm"])
    _pip(["git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt"])
    print("Install complete.")
else:
    print("Using current environment. Set FORCE_INSTALL=True to install here.")

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from engibench.utils.all_problems import BUILTIN_PROBLEMS
from engiopt.workshops.dcc26.notebook_helpers import (
    mean_pairwise_l2,
    show_feasibility_bars,
    show_objective_comparison,
    show_optimization_trajectories,
    show_pairwise_distance_heatmap,
)

SEED = 7

---
## 0 — *Load what Notebook 01 produced*

Notebook 01 saved three files into an artifacts folder:

- `generated_designs.npy` — the generator's outputs on held-out test scenarios.
- `baseline_designs.npy` — the optimiser's answers for those same scenarios.
- `conditions.json` — the scenarios themselves.

We also reload the EngiBench problem, because all of our evaluation logic is going to live on its methods.

In [ ]:
ARTIFACT_DIR = (
    Path("/content/dcc26_artifacts") if "google.colab" in sys.modules
    else Path("workshops/dcc26/artifacts")
)

required = ["generated_designs.npy", "baseline_designs.npy", "conditions.json"]
missing = [f for f in required if not (ARTIFACT_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {ARTIFACT_DIR}. Run Notebook 01 first, "
        "or re-run its export cell."
    )

gen_designs = np.load(ARTIFACT_DIR / "generated_designs.npy")
baseline_designs = np.load(ARTIFACT_DIR / "baseline_designs.npy")
with open(ARTIFACT_DIR / "conditions.json") as f:
    conditions = json.load(f)

problem = BUILTIN_PROBLEMS["beams2d"](seed=SEED)

print(f"Generated designs : {gen_designs.shape}")
print(f"Baseline designs  : {baseline_designs.shape}")
print(f"Scenarios         : {len(conditions)}  (keys = {list(conditions[0].keys())})")

---
## 1 — *Does the design obey the rules?*

This is the **feasibility** question. A generator can produce a beam that looks reasonable but quietly uses twice the material budget, or has density values outside the physical range, or breaks some solver-stability rule you didn't think to check. Low training loss gives you no protection against any of those.

In Notebook 00 we met `problem.check_constraints(design, config)`: it runs every `THEORY` and `IMPL` constraint the benchmark ships with and returns the ones that fired. Here we just call it once per generated design and count how many scenarios come back clean.

In [ ]:
def feasibility_count(designs, configs):
    feasible_flags = []
    for d, cfg in zip(designs, configs):
        # len(violations) == 0  means every constraint passed.
        violations = problem.check_constraints(design=d, config=cfg)
        feasible_flags.append(len(violations) == 0)
    return np.array(feasible_flags)


gen_feasible = feasibility_count(gen_designs, conditions)
base_feasible = feasibility_count(baseline_designs, conditions)

print(f"Generated feasible : {gen_feasible.sum()} / {len(gen_feasible)}  "
      f"({gen_feasible.mean()*100:.0f}%)")
print(f"Baseline feasible  : {base_feasible.sum()} / {len(base_feasible)}  "
      f"({base_feasible.mean()*100:.0f}%)")

In [ ]:
# Same numbers, but as a bar chart so the ratio is obvious at a glance.
show_feasibility_bars(pd.DataFrame({
    "gen_feasible": gen_feasible,
    "base_feasible": base_feasible,
}))

**How to read this.** The baseline is the optimiser's output — by construction it should pass almost every constraint. If the generator's bar is noticeably shorter than the baseline's, the model is learning *something that looks like a beam* but not *something that obeys the benchmark rules*. That's a real failure mode and no amount of prettier pictures will fix it — you need to change the loss, the conditioning, or the architecture.

Notice the asymmetry: a design that's infeasible is disqualified regardless of how good its objective looks. **Feasibility gates performance.** We check it first for that reason.

---
## 2 — *Does the design actually work?*

This is the **performance** question. Among the designs that passed feasibility, how stiff are they *really* — measured by the physics simulator, not by pixel loss?

In Notebook 00 we met `problem.simulate(design, config)`. It takes seconds on `beams2d` and returns the engineering objective (compliance — lower is better). We run it on each generated design *and* on the matching baseline design under the *same* scenario, so we can compare apples to apples.

In [ ]:
rows = []
for i, (g, b, cfg) in enumerate(zip(gen_designs, baseline_designs, conditions)):
    problem.reset(seed=SEED + i)
    g_obj = float(problem.simulate(g, config=cfg)[0])
    problem.reset(seed=SEED + i)
    b_obj = float(problem.simulate(b, config=cfg)[0])
    rows.append({
        "sample": i,
        "gen_obj": g_obj,
        "base_obj": b_obj,
        "gen_minus_base": g_obj - b_obj,
        "gen_feasible": bool(gen_feasible[i]),
        "base_feasible": bool(base_feasible[i]),
    })

results = pd.DataFrame(rows)
results.head()

The two plots below show the same numbers in two ways. The **histogram** asks: do the two objective distributions sit on top of each other, or has the generator shifted right (higher compliance = worse)? The **scatter** asks: for each individual scenario, is the generated design above or below the `y = x` diagonal? Points *below* the diagonal are scenarios where the generator *beat* the optimiser — a rare but real thing on a well-trained model.

In [ ]:
show_objective_comparison(results)

mean_gap = results["gen_minus_base"].mean()
win_rate = float((results["gen_obj"] < results["base_obj"]).mean())
print(f"Mean objective gap (gen − base): {mean_gap:+.1f}  "
      f"(positive = generator is worse on average)")
print(f"Generator beats baseline on    : {win_rate*100:.0f}% of scenarios")

**How to read this.** For a *simple* supervised-MSE generator like ours, a positive mean gap is expected — the optimiser is a very strong baseline, and ten epochs of MSE training won't catch it. The number we'd care about in a paper is *how big* that gap is relative to the typical objective value, whether it stays stable across re-training with different seeds, and whether a more sophisticated model (GAN, diffusion, …) closes it.

---
## 3 — *Is the model producing varied designs, or one design 24 times?*

This is the **diversity** question, and it's the one that doesn't need a physics call — it's a property of the generated set itself. A generative model that collapses to a single beam topology gets a low training loss (average over the dataset looks fine) but is useless for exploration.

The crudest-but-useful measure is *mean pairwise L2 distance* between all generated designs: average how different any two outputs are. We compute the same number for the baseline set as a sanity reference — the baseline comes from an optimiser run on diverse scenarios, so it naturally spreads out.

In [ ]:
gen_div = mean_pairwise_l2(gen_designs)
base_div = mean_pairwise_l2(baseline_designs)

print(f"Mean pairwise L2 — generated : {gen_div:.2f}")
print(f"Mean pairwise L2 — baseline  : {base_div:.2f}")
print(f"Ratio (gen / base)           : {gen_div / base_div:.2f}")

In [ ]:
# The heatmap makes collapse visible — a big dark block means a cluster of
# near-duplicates. A mostly-uniform warm plot means healthy variety.
show_pairwise_distance_heatmap(gen_designs)

**How to read this.** If the generator's diversity is ≳ the baseline's, the model is exploring at least as widely as the optimiser. If it's much lower — say under half — the model is partially collapsing, and the scatter plot in Part 2 is lying to you: "objective close to baseline" might mean "objective close to *one* baseline, because the model is always outputting that one beam."

This is why diversity has to be reported *alongside* the objective, not instead of it.

---
## 4 — *Does the generator actually speed up the optimiser?*

This is the question that, if the answer is yes, justifies the whole pipeline.

Recall the argument from Notebook 01: the optimiser is slow, the generator is fast, and the dream is to amortise. But there's a gentler version of the same dream — even if the generator's designs aren't quite optimal, they might be *a much better starting point* for the classical optimiser than a blank slate. If so, then running *(generator → optimiser)* gets you the optimiser's quality in a fraction of its normal iterations.

We test this with a tiny demo: pick three scenarios, feed the generator's design into `problem.optimize(...)` as the starting point, and watch the compliance curve. We compare where the trajectory *starts* (that's what the generator gave us for free) and where it *ends* (that's where the optimiser drives it) against the baseline's compliance for the same scenario.

> **Heads-up:** `problem.optimize(...)` runs the real FEM topology optimiser — each scenario takes ~30 seconds depending on hardware.

In [ ]:
N_WARMSTART_DEMO = 3

opt_data = []
for i in range(min(N_WARMSTART_DEMO, len(gen_designs))):
    cfg = dict(conditions[i])

    # Run the optimiser starting from the GENERATED design — this is the warmstart.
    problem.reset(seed=SEED + i)
    _, history = problem.optimize(gen_designs[i], config=cfg)
    trajectory = [float(step.obj_values[0]) for step in history]

    # The compliance of the ORIGINAL baseline design — the target to hit.
    problem.reset(seed=SEED + i)
    base_obj = float(problem.simulate(baseline_designs[i], config=cfg)[0])

    opt_data.append({
        "sample_idx": i,
        "obj_trajectory": trajectory,
        "base_obj": base_obj,
    })
    print(f"Sample {i}: start = {trajectory[0]:8.1f},  end = {trajectory[-1]:8.1f},  "
          f"baseline = {base_obj:8.1f}  ({len(trajectory)} optimiser steps)")

In [ ]:
show_optimization_trajectories(opt_data)

**How to read this.** The three numbers printed at the top of each panel are the **optimality gaps** a serious benchmark would report:

- **IOG** (Initial Optimality Gap) — how far the *generated starting point* is from the baseline. Small or negative means the model already gave the optimiser something close to the answer.
- **FOG** (Final Optimality Gap) — how far the *optimiser's output from that start* is from the baseline. Close to zero means the warmstart didn't trap the optimiser in a bad local minimum.
- **COG** (Cumulative Optimality Gap) — the shaded area. Small means the optimiser converged quickly from this start.

For our MSE generator, IOG is usually awful (pictures are blurry) but FOG recovers fast — the optimiser does its job. The interesting question for a better model is whether you can shrink *both*: a generator whose outputs are already near-optimal *and* stay there after a handful of optimiser steps would be a genuine speedup.

---
## Putting it together

Four questions, four numbers (or small sets of numbers). If you were writing a one-row benchmark table for a paper, this is essentially what would go in it:

In [ ]:
summary = pd.DataFrame([{
    "feasible %":              f"{gen_feasible.mean()*100:.0f}%",
    "mean obj gap (gen−base)": f"{results['gen_minus_base'].mean():+.1f}",
    "win rate vs baseline":    f"{(results['gen_obj'] < results['base_obj']).mean()*100:.0f}%",
    "diversity (L2)":          f"{gen_div:.2f}",
    "baseline diversity (L2)": f"{base_div:.2f}",
    "warmstart IOG (mean of demo)":  f"{np.mean([d['obj_trajectory'][0]  - d['base_obj'] for d in opt_data]):+.1f}",
    "warmstart FOG (mean of demo)":  f"{np.mean([d['obj_trajectory'][-1] - d['base_obj'] for d in opt_data]):+.1f}",
}]).T.rename(columns={0: "value"})
summary

Look at those seven numbers side by side. A row like this is what reviewers, collaborators, and future-you actually need — **not** a pretty grid of designs, and **not** a training loss. Every field answered a question the design-pictures-alone could not.

More importantly: every single field came from a method we already had in Notebook 00. No new infrastructure. The benchmark contract *is* the evaluation contract.

---
## What we deliberately skipped

A publication-grade evaluation would add, at minimum:

- **Distributional metrics** (MMD, DPP, FID, coverage/precision-recall) — do the *two distributions* of designs match, not just pairs?
- **Novelty against the training set** — is the model generalising, or quietly copy-pasting?
- **Multiple seeds** — any single training run lies; report mean ± std over 3–5 seeds.
- **Many more warmstart scenarios** — 3 is a demo, not a statistic.
- **Per-scenario breakdowns** — does the generator fail uniformly, or only on certain condition ranges?

Every one of those additions uses the *same* `problem` methods, just in different aggregations. The full pipeline in the companion notebook `02_evaluate_metrics.ipynb` walks through them.

---
## Reflect before moving on

1. Look at the seven-row summary. Which single number would change your mind the most if it were very good, or very bad? Why that one and not the others?
2. Suppose the feasibility rate is high but the mean objective gap is also high. What does that mean about the generator — is it being too *safe*, or too *dumb*, and how would you tell the difference?
3. The warmstart demo compared *generator-started* optimisation to a baseline. What would be a fairer comparison for claiming the generator *helps* the optimiser? (Hint: Notebook 00's `problem.optimize(start, cfg)` started from a uniform field.)

## Next

You've now seen the full workflow of benchmark-driven research in engineering design: **consume** a benchmark (Notebook 00), **train against** it (Notebook 01), and **evaluate on** it (this notebook). **Notebook 03** flips the perspective and shows what it takes to *build* a new EngiBench problem of your own — answering the same eight researcher's questions from Notebook 00, but in code you write yourself.